In [0]:
# ============================================================
# BRONZE TO SILVER: E-commerce Orders
# RetailPulse Data Pipeline
# ============================================================
# Reads raw e-commerce JSON files from Bronze layer
# Produces two normalised Delta tables in Silver:
#   - retailpulse.silver.dim_orders
#   - retailpulse.silver.fact_order_items
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

# ------------------------------------------------------------
# CELL 1: Read Bronze e-commerce JSON data
# ------------------------------------------------------------

bronze_path = "abfss://bronze@retailpulsedatalake.dfs.core.windows.net/ecommerce/"

# Read all files with recursiveFileLookup
df_raw = spark.read \
    .option("multiline", "true") \
    .option("inferSchema", "true") \
    .option("recursiveFileLookup", "true") \
    .json(bronze_path)

# Explode the orders array — each order becomes its own row
df_exploded = df_raw.select(F.explode(F.col("orders")).alias("order"))

# Flatten the nested struct into individual columns
df_flat = df_exploded.select("order.*")

print(f"Row count: {df_flat.count()}")
display(df_flat.limit(5))



Row count: 1678


customer_email,customer_id,delivery_address_city,device_type,discount_applied,fulfillment_type,line_item_id,line_total,order_id,order_status,order_time,order_total,product_id,product_name,quantity,shipping_cost,store_id,unit_price
a1efedc11aa0b7262a6411a44ed5a6ba7eec38ecdb8a503865f793ad6d799b1c,CUST_5970,Mutare,mobile,0.16,delivery,337418f5-8b56-4bee-b30c-8935216740ce_LI_001,48.77,337418f5-8b56-4bee-b30c-8935216740ce,shipped,2026-05-25T10:28:34,87.03,SKU_008,Chicken 1kg,7,5.58,ONLINE_DELIVERY,6.99
a1efedc11aa0b7262a6411a44ed5a6ba7eec38ecdb8a503865f793ad6d799b1c,CUST_5970,Mutare,mobile,1.28,delivery,337418f5-8b56-4bee-b30c-8935216740ce_LI_002,32.68,337418f5-8b56-4bee-b30c-8935216740ce,shipped,2026-05-25T10:28:34,87.03,SKU_002,Cooking Oil 2L,4,5.58,ONLINE_DELIVERY,8.49
3d448988c4f99aa88c4defd6526660a957a3017042afb82460b6611a1d5ac113,CUST_1022,Mutare,tablet,0.08,delivery,013fe748-f684-4363-ab29-9e272162e287_LI_001,7.39,013fe748-f684-4363-ab29-9e272162e287,returned,2026-05-25T22:14:42,69.41,SKU_005,Bread Loaf,3,5.3,ONLINE_DELIVERY,2.49
3d448988c4f99aa88c4defd6526660a957a3017042afb82460b6611a1d5ac113,CUST_1022,Mutare,tablet,0.02,delivery,013fe748-f684-4363-ab29-9e272162e287_LI_002,17.43,013fe748-f684-4363-ab29-9e272162e287,returned,2026-05-25T22:14:42,69.41,SKU_007,Eggs 6 pack,5,5.3,ONLINE_DELIVERY,3.49
3d448988c4f99aa88c4defd6526660a957a3017042afb82460b6611a1d5ac113,CUST_1022,Mutare,tablet,0.67,delivery,013fe748-f684-4363-ab29-9e272162e287_LI_003,39.29,013fe748-f684-4363-ab29-9e272162e287,returned,2026-05-25T22:14:42,69.41,SKU_003,Rice 5kg,4,5.3,ONLINE_DELIVERY,9.99


In [0]:
# ------------------------------------------------------------
# CELL 2: Clean and transform
# ------------------------------------------------------------

df_cleaned = df_flat \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("line_item_id").isNotNull()) \
    .filter(F.col("quantity") > 0) \
    .filter(F.col("line_total") > 0) \
    .withColumn("discount_applied",
        F.when(F.col("discount_applied").isNull(), 0)
        .otherwise(F.col("discount_applied"))
    ) \
    .withColumn("order_time_utc",
        F.to_utc_timestamp(
            F.to_timestamp(F.col("order_time")),
            "Africa/Harare"
        )
    ) \
    .drop("order_time")

# ------------------------------------------------------------
# CELL 3: Split into dim_orders and fact_order_items
# ------------------------------------------------------------

dim_orders = df_cleaned \
    .select(
        "order_id",
        "customer_id",
        "customer_email",
        "delivery_address_city",
        "fulfillment_type",
        "store_id",
        "shipping_cost",
        "order_total",
        "order_status",
        "device_type",
        "order_time_utc"
    ) \
    .distinct()

fact_order_items = df_cleaned \
    .select(
        "line_item_id",
        "order_id",
        "product_id",
        "product_name",
        "quantity",
        "unit_price",
        "discount_applied",
        "line_total"
    )

# ------------------------------------------------------------
# CELL 4: Write to Silver Delta tables
# ------------------------------------------------------------

dim_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailpulse.silver.dim_orders")

fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailpulse.silver.fact_order_items")

print("E-commerce Silver tables written successfully")



E-commerce Silver tables written successfully
